# 📚 01 — The Knowledge Base and the RAG Agent

The twelve notebooks in this course build an **evaluation harness and a release gate**.
The agent they measure — "Lumen Cloud", a fictional cloud-hosting support assistant — exists
only to be measured. This first notebook builds the part of it that answers questions from
documentation, because retrieval is where the most interesting failures live.

## Learning Objectives
In this notebook, you will learn:
1. **Multilingual chunking and indexing** — how ~40 help-center articles in English, Spanish
   and Hindi become a searchable Chroma collection
2. **Hybrid retrieval** — why dense-only search quietly fails on exact identifiers, and how
   a few lines of lexical scoring rescue it
3. **Cite-or-refuse** — an agent that grounds every claim in a retrieved chunk, and says
   "I don't know" when the corpus doesn't cover the question
4. **Fault switches** — deliberate, one-line breakage that later notebooks' metrics must catch

## Prerequisites
- `pip install -r requirements.txt` in this folder (see the pins in the setup cell)
- `OPENAI_API_KEY` in a `.env` file at the repository root
- No prior notebook — this is the entry point for course 03

---
## 🗺️ The concept

A RAG agent is three decisions in a trench coat: **what to retrieve**, **whether that is
enough**, and **what to say**. Most teaching material only instruments the third one. The
harness we build across this course scores all three, so keep the boundaries visible:

```
 user question
      |
      v
+-------------+     +------------------+     +---------------------+
|  retrieve   | --> |  enough to       | --> |  answer WITH        |
|  (hybrid)   |     |  answer?         | yes |  citations [KB-012] |
+-------------+     +------------------+     +---------------------+
      |                     | no
      |                     v
      |             +---------------------+
      |             |  REFUSE             |
      |             |  "not in our docs"  |
      |             +---------------------+
      v
 chunks + ids  -----> these become `retrieved_contexts`,
                      the input to every RAG metric in notebook 04
```

### Key Concepts:
- **Refusal is a correct answer.** A support bot that invents a refund window is worse than
  one that says it doesn't know. Notebook 03 makes at least a fifth of the golden set
  refusal cases for exactly this reason.
- **Citations are structured output, not decoration.** `[KB-012]` is machine-checkable; "as
  per our policy" is not. Notebook 04 scores citation accuracy because of this.
- **The corpus has a deliberate hole.** Hindi security articles are one thin stub. A suite
  that only reports an average will never see it — that is the lesson of notebook 10.

---
## ⚙️ Part 0: Setup

Versions are pinned because RAGAS and DeepEval rename things between minor releases. If you
change a pin, expect to re-verify the metric imports in notebooks 04–06.

In [1]:
# ============================================================================
# ENVIRONMENT SETUP: pinned versions, paths, API key
# ============================================================================
# Pins for this course (also in requirements.txt):
#   langchain-openai==1.6.2   chromadb==1.5.9   sentence-transformers==6.0.1
#   langgraph==1.2.11         ragas==0.4.3      deepeval==4.2.3
# langgraph is pinned, not floated: resolving ragas or mlflow can pull it back
# to 1.0.x, which drops langgraph.runtime.ExecutionInfo.

import os
import json
import re
import warnings
from collections import defaultdict

from dotenv import load_dotenv

warnings.filterwarnings("ignore")

# .env lives at the repository root, three levels up from this notebook.
load_dotenv(os.path.abspath(os.path.join("..", "..", ".env")))

DATA_DIR = os.path.abspath("data")
CHROMA_DIR = os.path.join(DATA_DIR, "chroma")
os.makedirs(CHROMA_DIR, exist_ok=True)

print("✅ Setup complete")
print("📁 data/   ->", DATA_DIR)
print("🔑 OPENAI_API_KEY present:", bool(os.getenv("OPENAI_API_KEY")))

✅ Setup complete
📁 data/   -> ~/Github_Repos/Agent_Evaluation_Demystified/courses/03-agentic-eval-harness/data
🔑 OPENAI_API_KEY present: True


### 0.1 🎛️ The fault switches

Every notebook in this course opens with a `FAULTS` dict, all `False`. The "Break it"
section flips one and re-runs. This is the whole point of the harness: a metric you have
never watched fail is a metric you do not yet trust.

| Fault | What it simulates |
|---|---|
| `dense_only_retrieval` | Someone deletes the lexical half of hybrid search |
| `empty_index` | A deploy ships with the vector store unpopulated |
| `citation_drift` | The model cites a real document that does not support the claim |
| `thin_hindi_docs` | A language segment quietly loses most of its coverage |

In [2]:
# ============================================================================
# FAULT SWITCHES: all off by default — the "Break it" section flips one
# ============================================================================
FAULTS = {
    "dense_only_retrieval": False,   # drop the lexical half of hybrid retrieval
    "empty_index": False,            # index nothing at all
    "citation_drift": False,         # cite a document that does not support the answer
    "thin_hindi_docs": False,        # keep only the first Hindi article per category
}

print("🎛️  Faults armed:", {k: v for k, v in FAULTS.items() if v} or "none (clean run)")

🎛️  Faults armed: none (clean run)


---
## 📝 Part 1: The corpus

Forty short help-center articles across five categories and three languages. They are
fictional, deliberately terse, and deliberately uneven — Hindi has a single thin security
stub where English has four full articles.

Each article carries an `id` like `KB-012`. That id is what the agent cites and what
notebook 04 checks citations against, so it has to be stable.

In [3]:
# ============================================================================
# HELP CENTER ARTICLES: (id, lang, category, title, body)
# ============================================================================
# Categories: refunds | plans | security | howto | account
# Coverage is intentionally uneven — see `hi` + `security` below.

ARTICLES = [
    # ---------------------------------------------------------------- English
    ("KB-001", "en", "refunds", "Refund eligibility window",
     "Lumen Cloud refunds any charge disputed within 30 days of the invoice date. "
     "Refunds are issued to the original payment method and settle in 5 to 7 business days. "
     "Charges older than 30 days are not eligible."),
    ("KB-002", "en", "refunds", "Duplicate and double charges",
     "If the same invoice is charged twice, the duplicate is always refunded in full and the "
     "30-day window does not apply. Report a duplicate charge with both transaction IDs."),
    ("KB-003", "en", "refunds", "Refunds on annual plans",
     "Annual plans are refunded pro rata for whole unused months. Partial months are not "
     "refunded. Downgrades mid-term produce account credit rather than a cash refund."),
    ("KB-004", "en", "refunds", "What is not refundable",
     "One-time setup fees, overage charges already consumed, and third-party marketplace "
     "add-ons are non-refundable. Taxes are refunded together with the underlying charge."),
    ("KB-005", "en", "plans", "Plan tiers and limits",
     "Starter includes 2 vCPU and 100 GB transfer. Growth includes 8 vCPU and 1 TB transfer. "
     "Scale includes 32 vCPU and unmetered transfer with fair-use review above 20 TB."),
    ("KB-006", "en", "plans", "Upgrading a plan",
     "Upgrades take effect immediately and are billed pro rata for the remainder of the "
     "current cycle. The new tier's limits apply from the moment the upgrade completes."),
    ("KB-007", "en", "plans", "Downgrading a plan",
     "Downgrades take effect at the end of the current billing cycle so you keep the higher "
     "limits you already paid for. A downgrade cannot be scheduled more than one cycle ahead."),
    ("KB-008", "en", "plans", "Overage pricing",
     "Transfer above the plan allowance is billed at 0.09 USD per GB on Starter and Growth. "
     "Scale has no transfer overage but is subject to fair-use review."),
    ("KB-009", "en", "security", "Two-factor authentication",
     "Lumen Cloud supports TOTP authenticator apps and hardware security keys. SMS codes were "
     "retired in 2024. Enabling 2FA signs out all other active sessions."),
    ("KB-010", "en", "security", "Password reset policy",
     "A reset link is valid for 60 minutes and can be used once. Support staff can trigger a "
     "reset but can never read or choose a password on a customer's behalf."),
    ("KB-011", "en", "security", "Data residency",
     "Data is stored in the region chosen at account creation: eu-west, us-east or ap-south. "
     "Region cannot be changed after creation; migration requires a new account."),
    ("KB-012", "en", "security", "Incident disclosure",
     "Confirmed security incidents affecting customer data are disclosed to affected accounts "
     "within 72 hours, with a public post-mortem within 14 days."),
    ("KB-013", "en", "howto", "Creating an API token",
     "Generate tokens under Settings then API. A token is shown once at creation and cannot be "
     "retrieved later. Tokens inherit the permissions of the user that created them."),
    ("KB-014", "en", "howto", "Connecting a custom domain",
     "Add the domain in the dashboard, then create the displayed CNAME record with your DNS "
     "provider. Verification usually completes within 30 minutes of the record propagating."),
    ("KB-015", "en", "howto", "Reading your invoice",
     "An invoice lists the base plan charge, metered overage, taxes, and any credits applied. "
     "Credits are always applied before the payment method is charged."),
    ("KB-016", "en", "howto", "Exporting usage data",
     "Usage can be exported as CSV for any period up to 13 months. Exports over 90 days are "
     "generated asynchronously and emailed as a download link."),
    ("KB-017", "en", "account", "Changing the account owner",
     "Ownership transfer requires confirmation from both the current and the incoming owner. "
     "The incoming owner must already be a member of the account."),
    ("KB-018", "en", "account", "Adding team members",
     "Members are invited by email and assigned one of four roles: owner, admin, developer or "
     "billing. Only owners and admins can invite further members."),
    ("KB-019", "en", "account", "Closing an account",
     "Closing an account schedules deletion 30 days out. During those 30 days the account can "
     "be restored in full. After 30 days, deletion is irreversible."),
    ("KB-020", "en", "account", "Updating billing contacts",
     "Billing contacts receive invoices and dunning notices. An account may have up to five "
     "billing contacts, and at least one is always required."),

    # ---------------------------------------------------------------- Spanish
    ("KB-021", "es", "refunds", "Plazo para solicitar un reembolso",
     "Lumen Cloud reembolsa cualquier cargo reclamado dentro de los 30 días posteriores a la "
     "fecha de la factura. El reembolso se emite al método de pago original y tarda de 5 a 7 "
     "días hábiles."),
    ("KB-022", "es", "refunds", "Cargos duplicados",
     "Si una misma factura se cobra dos veces, el cargo duplicado se reembolsa íntegramente y "
     "el plazo de 30 días no se aplica."),
    ("KB-023", "es", "refunds", "Qué no es reembolsable",
     "Las cuotas de alta, los excedentes ya consumidos y los complementos de terceros no son "
     "reembolsables. Los impuestos se reembolsan junto con el cargo correspondiente."),
    ("KB-024", "es", "plans", "Niveles de plan y límites",
     "Starter incluye 2 vCPU y 100 GB de transferencia. Growth incluye 8 vCPU y 1 TB. Scale "
     "incluye 32 vCPU y transferencia sin medir, sujeta a revisión de uso razonable."),
    ("KB-025", "es", "plans", "Mejorar de plan",
     "Las mejoras se aplican de inmediato y se facturan de forma prorrateada por el resto del "
     "ciclo actual."),
    ("KB-026", "es", "plans", "Bajar de plan",
     "Las bajadas de plan se aplican al final del ciclo de facturación actual, de modo que "
     "conserva los límites superiores que ya pagó."),
    ("KB-027", "es", "security", "Autenticación de dos factores",
     "Se admiten aplicaciones TOTP y llaves de seguridad físicas. Los códigos por SMS se "
     "retiraron en 2024. Activar 2FA cierra todas las demás sesiones activas."),
    ("KB-028", "es", "security", "Restablecimiento de contraseña",
     "El enlace de restablecimiento es válido durante 60 minutos y de un solo uso. El soporte "
     "nunca puede leer ni elegir una contraseña por usted."),
    ("KB-029", "es", "howto", "Crear un token de API",
     "Genere los tokens en Configuración y luego API. El token se muestra una sola vez al "
     "crearlo y no puede recuperarse después."),
    ("KB-030", "es", "howto", "Leer su factura",
     "La factura detalla el cargo del plan base, los excedentes medidos, los impuestos y los "
     "créditos aplicados. Los créditos se aplican antes de cobrar el método de pago."),
    ("KB-031", "es", "account", "Añadir miembros del equipo",
     "Los miembros se invitan por correo electrónico con uno de cuatro roles: propietario, "
     "administrador, desarrollador o facturación."),
    ("KB-032", "es", "account", "Cerrar una cuenta",
     "El cierre programa la eliminación a 30 días. Durante ese plazo la cuenta puede "
     "restaurarse por completo; después, la eliminación es irreversible."),

    # ------------------------------------------------------------------ Hindi
    # NOTE: `security` has a single thin stub here, against four English articles.
    # That gap is deliberate. Notebook 10 is about reporting that finds it.
    ("KB-033", "hi", "refunds", "रिफ़ंड की समय-सीमा",
     "इनvoice की तारीख़ से 30 दिनों के भीतर विवादित किसी भी शुल्क का रिफ़ंड किया जाता है। "
     "रिफ़ंड मूल भुगतान माध्यम में 5 से 7 कार्यदिवसों में पहुँचता है।"),
    ("KB-034", "hi", "refunds", "दोहरा शुल्क",
     "यदि एक ही इनvoice पर दो बार शुल्क लिया गया है, तो दोहरा शुल्क पूरा वापस किया जाता है "
     "और 30 दिन की सीमा लागू नहीं होती।"),
    ("KB-035", "hi", "plans", "प्लान स्तर और सीमाएँ",
     "Starter में 2 vCPU और 100 GB ट्रांसफ़र है। Growth में 8 vCPU और 1 TB है। Scale में "
     "32 vCPU और बिना माप वाला ट्रांसफ़र है।"),
    ("KB-036", "hi", "plans", "प्लान अपग्रेड करना",
     "अपग्रेड तुरंत लागू होता है और मौजूदा चक्र के शेष हिस्से के लिए आनुपातिक रूप से बिल किया जाता है।"),
    ("KB-037", "hi", "security", "दो-चरणीय प्रमाणीकरण",
     "TOTP ऐप समर्थित हैं।"),  # <- the thin stub: one sentence, no key rotation, no SMS note
    ("KB-038", "hi", "howto", "API टोकन बनाना",
     "Settings फिर API में जाकर टोकन बनाएँ। टोकन केवल एक बार दिखाया जाता है और बाद में "
     "प्राप्त नहीं किया जा सकता।"),
    ("KB-039", "hi", "howto", "अपना इनvoice पढ़ना",
     "इनvoice में बेस प्लान शुल्क, मापित अतिरिक्त उपयोग, कर और लागू क्रेडिट दिखाए जाते हैं।"),
    ("KB-040", "hi", "account", "टीम सदस्य जोड़ना",
     "सदस्यों को ईमेल से आमंत्रित किया जाता है और चार भूमिकाओं में से एक दी जाती है।"),
]

print("📚 Articles loaded:", len(ARTICLES))

📚 Articles loaded: 40


In [4]:
# ============================================================================
# COVERAGE MAP: language x category, before any chunking
# ============================================================================
# Counting the corpus by segment BEFORE indexing is the cheapest evaluation in
# this entire course, and it already finds the hole.

coverage = defaultdict(int)
for art_id, lang, category, title, body in ARTICLES:
    coverage[(lang, category)] += 1

categories = ["refunds", "plans", "security", "howto", "account"]
print("%-8s %s" % ("lang", "  ".join("%-9s" % c for c in categories)))
for lang in ["en", "es", "hi"]:
    row = "  ".join("%-9d" % coverage[(lang, c)] for c in categories)
    print("%-8s %s" % (lang, row))

print()
print("⚠️  hi/security has 1 article against en/security's 4 — remember this number.")

lang     refunds    plans      security   howto      account  
en       4          4          4          4          4        
es       3          3          2          2          2        
hi       2          2          1          2          1        

⚠️  hi/security has 1 article against en/security's 4 — remember this number.


---
## ✂️ Part 2: Chunking

These articles are short enough that one chunk each would work. We chunk anyway, with
overlap, because the thing we want to teach is what retrieval metrics see: **chunks, not
documents**. A chunk carries its parent article's id so a citation can always be resolved
back to something a human can read.

In [5]:
# ============================================================================
# CHUNKING: fixed-size character windows with overlap
# ============================================================================
# Deliberately simple. Sentence- or token-aware splitters are better in
# production; they would also hide what the overlap parameter does.

CHUNK_SIZE = 320      # characters — short articles, so this is ~1-2 sentences
CHUNK_OVERLAP = 60    # characters repeated between neighbours, so a fact split
                      # across a boundary still appears whole in one chunk


def chunk_text(text, size, overlap):
    """Split `text` into overlapping windows. Returns a list of strings."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + size
        chunks.append(text[start:end].strip())
        if end >= len(text):
            break
        start = end - overlap
    return chunks


def build_chunks(articles):
    """Turn (id, lang, category, title, body) rows into flat chunk dicts."""
    rows = []
    for art_id, lang, category, title, body in articles:
        full = title + ". " + body
        for i, piece in enumerate(chunk_text(full, CHUNK_SIZE, CHUNK_OVERLAP)):
            rows.append({
                "chunk_id": art_id + "#" + str(i),
                "article_id": art_id,
                "lang": lang,
                "category": category,
                "title": title,
                "text": piece,
            })
    return rows


CHUNKS = build_chunks(ARTICLES)

print("✂️  Chunks:", len(CHUNKS), "from", len(ARTICLES), "articles")
print("📄 Example:", CHUNKS[0]["chunk_id"], "->", CHUNKS[0]["text"][:90], "...")

✂️  Chunks: 40 from 40 articles
📄 Example: KB-001#0 -> Refund eligibility window. Lumen Cloud refunds any charge disputed within 30 days of the i ...


---
## 🧮 Part 3: Embedding and the Chroma index

`paraphrase-multilingual-MiniLM-L12-v2` maps all three languages into one vector space, so
a Hindi question can retrieve an English article. That is a feature and a trap: it makes the
thin Hindi corpus *look* fine, because the English articles quietly cover for it. Notebook 04
scores answers per language to expose that.

In [6]:
# ============================================================================
# EMBEDDING MODEL: multilingual, small, runs locally on CPU
# ============================================================================
from sentence_transformers import SentenceTransformer

EMBED_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
embedder = SentenceTransformer(EMBED_MODEL_NAME)


def embed(texts):
    """Embed a list of strings. normalize=True so cosine == dot product."""
    vectors = embedder.encode(texts, normalize_embeddings=True)
    return [list(map(float, v)) for v in vectors]   # plain lists, no numpy downstream


print("🤖 Embedding model:", EMBED_MODEL_NAME)
print("📐 Dimensions:", len(embed(["probe"])[0]))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

🤖 Embedding model: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
📐 Dimensions: 384


In [7]:
# ============================================================================
# BUILD INDEX: persist a Chroma collection under data/chroma/
# ============================================================================
import chromadb

COLLECTION_NAME = "lumen_kb"

client = chromadb.PersistentClient(path=CHROMA_DIR)

# Rebuild from scratch so re-running the notebook is idempotent.
try:
    client.delete_collection(COLLECTION_NAME)
except Exception:
    pass   # first run: nothing to delete

collection = client.create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"},   # cosine, to match normalized embeddings
)


# What each collection actually contains, keyed by collection name. The lexical
# half of retrieval reads this, so "the index is empty" means empty for BOTH
# halves — which is what an index really is. Keying it by collection means
# switching `collection` switches the corpus with it, so the fault demos below
# need no manual save-and-restore.
INDEXED = {}


def index_chunks(coll, chunks):
    """Embed and add chunks to a Chroma collection, and record what went in."""
    INDEXED[coll.name] = list(chunks)
    if not chunks:
        print("⚠️  Nothing to index — the collection will stay empty.")
        return
    coll.add(
        ids=[c["chunk_id"] for c in chunks],
        documents=[c["text"] for c in chunks],
        embeddings=embed([c["text"] for c in chunks]),
        metadatas=[
            {"article_id": c["article_id"], "lang": c["lang"],
             "category": c["category"], "title": c["title"]}
            for c in chunks
        ],
    )


index_chunks(collection, CHUNKS)

print("✅ Indexed", collection.count(), "chunks into", COLLECTION_NAME)
print("💾 Persisted to", CHROMA_DIR)

✅ Indexed 40 chunks into lumen_kb
💾 Persisted to ~/Github_Repos/Agent_Evaluation_Demystified/courses/03-agentic-eval-harness/data/chroma


---
## 🔎 Part 4: Hybrid retrieval

Dense search is good at paraphrase and bad at identifiers. Ask about `KB-012` or "eu-west"
and an embedding model will happily return something *topically* adjacent. A few lines of
token overlap fixes it, and the `dense_only_retrieval` fault deletes those lines so you can
watch the failure arrive in notebook 04's context-recall numbers.

In [8]:
# ============================================================================
# HYBRID RETRIEVAL: dense (Chroma) + lexical (token overlap), merged by rank
# ============================================================================

def lexical_scores(query, chunks):
    """Score chunks by how many query tokens they contain. Plain and transparent."""
    q_tokens = set(re.findall(r"\w+", query.lower()))
    scored = []
    for c in chunks:
        c_tokens = set(re.findall(r"\w+", c["text"].lower()))
        overlap = len(q_tokens & c_tokens)
        if overlap:
            scored.append((overlap / max(len(q_tokens), 1), c))
    scored.sort(key=lambda pair: pair[0], reverse=True)
    return scored


def retrieve(query, k=4):
    """Return up to k chunk dicts, most relevant first.

    Reciprocal-rank fusion: a chunk that both halves rank highly beats a chunk
    that only one half loves. RRF_CONSTANT damps the top-rank advantage.
    """
    RRF_CONSTANT = 60

    # Both halves read the SAME corpus: whatever was actually indexed into this
    # collection. Reading the global CHUNKS here instead would let the lexical
    # half answer from documents the index does not have.
    corpus = INDEXED.get(collection.name, [])

    dense_hits = collection.query(query_embeddings=embed([query]), n_results=k * 2)
    dense_ids = dense_hits["ids"][0] if dense_hits["ids"] else []

    if FAULTS["dense_only_retrieval"]:
        lexical_ids = []                      # <- the fault: lexical half removed
    else:
        lexical_ids = [c["chunk_id"] for _, c in lexical_scores(query, corpus)[:k * 2]]

    fused = defaultdict(float)
    for rank, chunk_id in enumerate(dense_ids):
        fused[chunk_id] += 1.0 / (RRF_CONSTANT + rank)
    for rank, chunk_id in enumerate(lexical_ids):
        fused[chunk_id] += 1.0 / (RRF_CONSTANT + rank)

    by_id = {c["chunk_id"]: c for c in corpus}
    ranked = sorted(fused.items(), key=lambda pair: pair[1], reverse=True)
    return [by_id[chunk_id] for chunk_id, _ in ranked[:k] if chunk_id in by_id]


hits = retrieve("How long do I have to dispute a charge?")
print("🔍 Query: How long do I have to dispute a charge?")
for h in hits:
    print("   ", h["article_id"], "|", h["lang"], "|", h["title"])

🔍 Query: How long do I have to dispute a charge?
    KB-001 | en | Refund eligibility window
    KB-002 | en | Duplicate and double charges
    KB-020 | en | Updating billing contacts
    KB-034 | hi | दोहरा शुल्क


---
## 🤖 Part 5: The cite-or-refuse agent

The prompt does three jobs, and each one exists so a later metric has something to grab:

| Prompt instruction | Metric that checks it |
|---|---|
| "Answer only from the context" | Faithfulness — notebook 04 |
| "Cite the article id in square brackets" | Citation accuracy — notebook 04 |
| "If the context does not cover it, refuse" | Refusal correctness — notebook 04 |

> **Note**: `ChatOpenAI` is the default here because the rest of this repository runs on
> `OPENAI_API_KEY` alone. The commented alternatives are one-line switches.

In [9]:
# ============================================================================
# MODEL SETUP: OpenAI by default, one-line switch to a local or hosted peer
# ============================================================================
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
# from langchain_ollama import ChatOllama
# llm = ChatOllama(model="llama3.1", temperature=0)          # fully local, no API key
# from langchain_anthropic import ChatAnthropic
# llm = ChatAnthropic(model="claude-sonnet-5", temperature=0)

print("🤖 Chat model ready:", llm.model_name)

🤖 Chat model ready: gpt-4o-mini


In [10]:
# ============================================================================
# THE RAG AGENT: retrieve -> ground -> cite, or refuse
# ============================================================================

REFUSAL_TEXT = "I don't have documentation covering that, so I can't answer it reliably."

SYSTEM_PROMPT = """You are Lumen Cloud's documentation assistant.

Rules, in priority order:
1. Answer ONLY from the CONTEXT below. Never use outside knowledge.
2. Cite the article id of every claim in square brackets, like [KB-001].
3. If the CONTEXT does not contain the answer, reply with exactly this sentence
   and nothing else: "{refusal}"
4. Reply in the same language the user wrote in.

CONTEXT:
{context}"""


def format_context(chunks):
    """Render retrieved chunks so the model can cite them by id."""
    return "\n\n".join("[" + c["article_id"] + "] " + c["text"] for c in chunks)


def rag_agent(question, k=4):
    """Answer `question` from the knowledge base. Returns a trace dict.

    The return value is shaped for evaluation, not for display: `retrieved_contexts`
    and `citations` are exactly what notebook 04's RAGAS metrics consume.
    """
    chunks = retrieve(question, k=k)

    if FAULTS["citation_drift"] and len(chunks) > 1:
        # The fault: shuffle the context so cited ids no longer match the text
        # that actually supports the answer. The answer still LOOKS sourced.
        chunks = chunks[1:] + chunks[:1]

    context = format_context(chunks)
    messages = [
        {"role": "system",
         "content": SYSTEM_PROMPT.format(refusal=REFUSAL_TEXT, context=context)},
        {"role": "user", "content": question},
    ]
    answer = llm.invoke(messages).content.strip()

    return {
        "question": question,
        "answer": answer,
        "retrieved_contexts": [c["text"] for c in chunks],
        "retrieved_ids": [c["article_id"] for c in chunks],
        "citations": sorted(set(re.findall(r"\[(KB-\d+)\]", answer))),
        "refused": answer.startswith(REFUSAL_TEXT[:40]),
        "langs_retrieved": sorted(set(c["lang"] for c in chunks)),
    }


print("✅ rag_agent() defined — retrieve, ground, cite or refuse")

✅ rag_agent() defined — retrieve, ground, cite or refuse


---
## ▶️ Part 6: See it work

Three questions, chosen to exercise three different behaviours: a fact that is squarely in
the corpus, a question asked in Hindi, and a question the corpus genuinely does not answer.
The third one should refuse. If it doesn't, everything downstream in this course is measuring
a bot that makes things up.

In [11]:
# ============================================================================
# SEE IT WORK: covered, cross-lingual, and absent
# ============================================================================

PROBE_QUESTIONS = [
    ("covered",       "I was charged twice for the same invoice. Do I get all of it back?"),
    ("cross-lingual", "क्या मैं अपना प्लान अपग्रेड कर सकता हूँ और बिल कैसे लगेगा?"),
    ("absent",        "Does Lumen Cloud offer a student discount?"),
]

for label, question in PROBE_QUESTIONS:
    trace = rag_agent(question)
    print("=" * 78)
    print("🔍 [%s] %s" % (label, question))
    print("📄 retrieved:", ", ".join(trace["retrieved_ids"]))
    print("🤖", trace["answer"][:300])
    print("📋 citations:", trace["citations"], "| refused:", trace["refused"])

🔍 [covered] I was charged twice for the same invoice. Do I get all of it back?
📄 retrieved: KB-002, KB-001, KB-034, KB-022
🤖 Yes, if you were charged twice for the same invoice, the duplicate charge will be refunded in full and the 30-day window does not apply. You should report the duplicate charge with both transaction IDs [KB-002].
📋 citations: ['KB-002'] | refused: False
🔍 [cross-lingual] क्या मैं अपना प्लान अपग्रेड कर सकता हूँ और बिल कैसे लगेगा?
📄 retrieved: KB-036, KB-039, KB-007, KB-006
🤖 हाँ, आप अपना प्लान अपग्रेड कर सकते हैं। अपग्रेड तुरंत लागू होता है और मौजूदा चक्र के शेष हिस्से के लिए आनुपातिक रूप से बिल किया जाता है [KB-036][KB-006]।
📋 citations: ['KB-006', 'KB-036'] | refused: False
🔍 [absent] Does Lumen Cloud offer a student discount?
📄 retrieved: KB-001, KB-021, KB-009, KB-002
🤖 I don't have documentation covering that, so I can't answer it reliably.
📋 citations: [] | refused: True


---
## 💥 Part 7: Break it

Two faults, two different shapes of failure. Notice that **neither one raises an exception**.
That is the entire reason this course exists: a broken RAG agent returns confident prose with
a 200 status code, and only a metric tells you otherwise.

In [12]:
# ============================================================================
# BREAK IT (1): empty_index — the deploy that shipped without data
# ============================================================================
# Build a throwaway collection so the persisted index stays clean for notebooks
# 02-12. Poisoning a shared artifact from a fault demo is its own kind of bug.

FAULTS["empty_index"] = True

broken = client.get_or_create_collection(name="lumen_kb_broken")
index_chunks(broken, [])                      # the fault: index nothing

healthy_collection = collection               # keep a handle on the good one
collection = broken                           # point retrieve() at the empty index

trace = rag_agent("How long do I have to dispute a charge?")
print("🤖", trace["answer"][:220])
print("📄 retrieved:", trace["retrieved_ids"] or "NOTHING")
print("📋 refused:", trace["refused"])
print()
print("⚠️  The agent refuses everything. Availability looks perfect; usefulness is zero.")
print("    Notebook 12's alert for this is 'citation-free answer rate', not an error rate.")

collection = healthy_collection               # restore
FAULTS["empty_index"] = False

⚠️  Nothing to index — the collection will stay empty.
🤖 I don't have documentation covering that, so I can't answer it reliably.
📄 retrieved: NOTHING
📋 refused: True

⚠️  The agent refuses everything. Availability looks perfect; usefulness is zero.
    Notebook 12's alert for this is 'citation-free answer rate', not an error rate.


In [16]:
# ============================================================================
# BREAK IT (2): thin_hindi_docs — a language segment loses coverage
# ============================================================================
# Keep only the FIRST Hindi article per category. English and Spanish untouched.

FAULTS["thin_hindi_docs"] = True

seen = set()
thinned = []
for art in ARTICLES:
    art_id, lang, category, title, body = art
    if lang == "hi":
        if (lang, category) in seen:
            continue                          # the fault: drop the rest
        seen.add((lang, category))
    thinned.append(art)

print("📚 Articles: %d -> %d (dropped %d Hindi)" % (
    len(ARTICLES), len(thinned), len(ARTICLES) - len(thinned)))

# Re-chunk and re-index into the throwaway collection, then ask in Hindi.
CHUNKS_FULL = CHUNKS
CHUNKS = build_chunks(thinned)
try:
    client.delete_collection("lumen_kb_broken")
except Exception:
    pass
broken = client.create_collection(name="lumen_kb_broken",
                                  metadata={"hnsw:space": "cosine"})
index_chunks(broken, CHUNKS)
collection = broken

trace = rag_agent("दो-चरणीय प्रमाणीकरण के लिए कौन से तरीके समर्थित हैं?")
print()
print("🤖", trace["answer"][:300])
print("📄 retrieved:", trace["retrieved_ids"])
print("🌐 languages retrieved:", trace["langs_retrieved"])
print()
print("⚠️  The answer is probably still GOOD — English articles covered for the gap.")
print("    Aggregate faithfulness will not move. Only a per-language report sees this,")
print("    which is what notebook 10 builds and notebook 11 gates on.")

CHUNKS = CHUNKS_FULL
collection = healthy_collection
FAULTS["thin_hindi_docs"] = False

📚 Articles: 40 -> 37 (dropped 3 Hindi)

🤖 Lumen Cloud दो-चरणीय प्रमाणीकरण के लिए TOTP ऐप और हार्डवेयर सुरक्षा कुंजी का समर्थन करता है। [KB-009]
📄 retrieved: ['KB-037', 'KB-009', 'KB-033', 'KB-027']
🌐 languages retrieved: ['en', 'es', 'hi']

⚠️  The answer is probably still GOOD — English articles covered for the gap.
    Aggregate faithfulness will not move. Only a per-language report sees this,
    which is what notebook 10 builds and notebook 11 gates on.


---
## 🧪 Part 8: Try it yourself

**Exercise 1 — Make the identifier failure visible.**
Set `FAULTS["dense_only_retrieval"] = True` and ask *"Which article covers data residency
regions?"* and *"What does KB-011 say?"*. One of those degrades far more than the other.
Explain why in one sentence, then turn the fault off.

**Exercise 2 — Find the refusal boundary.**
`rag_agent()` refuses when the model judges the context insufficient — there is no explicit
score threshold. Write five questions that sit near the edge of the corpus (partially
covered, adjacent topic, wrong product). How many refuse? Keep the list: notebook 03 wants
exactly these cases, because the boundary is where a golden set earns its keep.

---
## 📝 Recap and interview angle

### What this notebook built
- **A multilingual Chroma index** over 40 articles, persisted to `data/chroma/`
- **Hybrid retrieval** — dense plus lexical, fused by reciprocal rank
- **A cite-or-refuse agent** returning a trace dict shaped for evaluation, not for display
- **Four fault switches** that break it quietly, the way production breaks

### Produces, for later notebooks
| Artifact | Consumed by |
|---|---|
| `data/chroma/` | notebooks 02, 04 |
| `data/kb_articles.json` | notebook 03 (writing golden cases against real article ids) |

### The interview angle
> *"How would you evaluate a RAG system?"*

The weak answer names metrics. The strong answer starts one step earlier: **what does the
system return that is checkable?** This notebook's agent returns `retrieved_contexts`,
`retrieved_ids`, `citations` and `refused` — four machine-readable fields. Every metric in
notebook 04 is possible because of that shape, and would be impossible if the agent returned
only a string.

The follow-up worth being ready for is the multilingual one. A shared embedding space means a
Hindi question can be answered from an English document, so a thin Hindi corpus produces
**good answers and a bad system** — the user cannot read their own documentation. No
aggregate metric catches it. That is the argument for stratified reporting, and you can make
it concrete with the number from the coverage map above: 1 article versus 4.

### Next
**[`02_tools_agents_and_supervisor_graph.ipynb`](02_tools_agents_and_supervisor_graph.ipynb)** —
SQLite, `@tool` functions with an approval gate, the supervisor routing graph, and the
trajectory dict that every level-2 and level-3 metric in this course consumes.

In [17]:
# ============================================================================
# SAVE ARTIFACTS: hand off to notebook 03 through a file, never an import
# ============================================================================
# Course convention: notebooks communicate through data/, so any notebook can be
# re-run on its own without executing the ones before it.

kb_path = os.path.join(DATA_DIR, "kb_articles.json")
with open(kb_path, "w", encoding="utf-8") as f:
    json.dump(
        [{"article_id": a, "lang": l, "category": c, "title": t, "body": b}
         for a, l, c, t, b in ARTICLES],
        f, ensure_ascii=False, indent=2,
    )

print("💾 Wrote", kb_path)
print("💾 Chroma collection '%s' persisted with %d chunks" % (
    COLLECTION_NAME, healthy_collection.count()))
print("✅ Notebook 01 complete — next: 02_tools_agents_and_supervisor_graph.ipynb")

💾 Wrote ~/Github_Repos/Agent_Evaluation_Demystified/courses/03-agentic-eval-harness/data/kb_articles.json
💾 Chroma collection 'lumen_kb' persisted with 40 chunks
✅ Notebook 01 complete — next: 02_tools_agents_and_supervisor_graph.ipynb
